# 4.5 章节实践（GEMM 实现与调优实战）

## 概述

前面已经完成基础 GEMM、流水排布和 Persistent 任务遍历。本节继续使用 `(M,N,K)=(512,36864,256)`，按四个阶段实践：扩展到实际 36 核并建立基线，开启双 buffer，调整 BM，最后结合左矩阵 A 常驻 L1 与 Persistent 列优先任务分配进行优化。每个阶段都在前一阶段的基础上验证正确性和性能，观察耗时、带宽和算力利用率的变化。

## 学习目标

- 根据 36 核推导输出任务分配，验证完整输出并建立性能基线。
- 通过 tiling 与左矩阵复用减少重复搬运，理解内存容量与性能之间的关系。
- 使用同一用例比较 kernel 耗时、有效带宽和算力利用率。

[本章导航](README.md) · [上一节](04.04_persistent.ipynb)

<p><small>执行前完成第1.4节环境准备，按本章README的统一入口运行。每页使用新内核；NPU程序由独立子进程执行，notebook内核不初始化设备。</small></p>

![36 核实践的四阶段路线](images/04.05_progression.svg)

## 阶段一：36 核实现与性能基线

A 的 shape 为 `(512,256)`，B 为 `(36864,256)`，输出 C 为 `(512,36864)`。为展示调优过程，本实践从 `BM=64、BN=128、stages=1` 起步；这是合法的教学起点，不是最优参数。

输出网格为 `8×288=2304` 个 tile。36 个核各承担 64 个完整输出块：`idx=bx×64+it`，`mi=idx//288`，`ni=idx%288`。

- `bx`：逻辑核任务编号，取值为 0～35。
- `it`：该核处理输出块的循环编号，取值为 0～63。
- `idx`：全局输出块编号。
- `mi`、`ni`：行块和列块编号。

每轮读入完整 K 的输入块，计算后写回独立输出。

下面是完整可执行代码，沿用连续任务分配。后续双 buffer 和 BM 调整使用同一份实现，只改变 BM、BN 和 stages。

<details>
<summary>展开36核完整实现（沿用前面的GEMM主体）</summary>

```python
import tilelang
import tilelang.language as T

BLOCKS = 36


def check_config(M, N, K, BM, BN, stages):
    if any(type(x) is not int or x <= 0 for x in (M, N, K, BM, BN, stages)):
        raise ValueError('Dimensions and stages must be positive integers')
    if BM not in (64, 128, 256) or BN not in (64, 128, 256) or K not in (64, 128, 256):
        raise ValueError('Teaching domain: BM/BN=64/128/256; full K=64/128/256')
    if stages not in (1, 2, 3) or M % BM or N % BN:
        raise ValueError('Stages=1/2/3; output element tails are not implemented')
    if (M // BM) * (N // BN) % BLOCKS:
        raise ValueError('This comparison uses equal complete tasks on 36 cores')
    if stages * 2 * (BM + BN) * K > 512 * 1024:
        raise ValueError('L1 buffer versions exceed the 512 KiB budget')
    if stages * 4 * BM * BN > 256 * 1024:
        raise ValueError('L0C buffer versions exceed the 256 KiB budget')


def gemm(M=512, N=36864, K=256, BM=128, BN=128, stages=1, persistent=False):
    check_config(M, N, K, BM, BN, stages)
    tasks = (M // BM) * (N // BN)
    blocks = BLOCKS
    s = stages
    mode = 'persistent' if persistent else 'contiguous'

    @T.prim_func
    def main(
        A: T.Tensor((M, K), 'bfloat16'),
        B: T.Tensor((N, K), 'bfloat16'),
        C: T.Tensor((M, N), 'float32'),
    ):
        with T.Kernel(blocks) as bx:
            a = T.alloc_l1((BM, K), 'bfloat16')
            b = T.alloc_l1((BN, K), 'bfloat16')
            c = T.alloc_l0c((BM, BN), 'float32')
            T.annotate_buffer_versions({a: s, b: s, c: s})
            if mode == 'persistent':
                for idx in T.Persistent([tasks], blocks, bx, num_stages=s):
                    mi = idx // (N // BN)
                    ni = idx % (N // BN)
                    T.copy(A[mi * BM : (mi + 1) * BM, :], a)
                    T.copy(B[ni * BN : (ni + 1) * BN, :], b)
                    T.gemm(a, b, c, transpose_B=True, clear_accum=True)

                    T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])
            else:
                for it in T.Pipelined(tasks // blocks, num_stages=s):
                    idx = bx * (tasks // blocks) + it
                    mi = idx // (N // BN)
                    ni = idx % (N // BN)
                    T.copy(A[mi * BM : (mi + 1) * BM, :], a)
                    T.copy(B[ni * BN : (ni + 1) * BN, :], b)
                    T.gemm(a, b, c, transpose_B=True, clear_accum=True)

                    T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])

    return main
```

</details>
阶段一的调用代码：

```python
main = gemm(M=512, N=36864, K=256, BM=64, BN=128, stages=1)
kernel = tilelang.compile(main, out_idx=-1)
```

先验证 random、zero、constant、pattern 四类输入，每类连续调用两次，对全部输出使用 `rtol=atol=1e-4` 比较。再采集三批 kernel 耗时，每批预热 10 次、保留 30 次，共 90 个有效样本。

```bash
source src/env.sh
msprof --output="${COURSE_OUTPUT_DIR}/practice_step1" --task-time=on --ai-core=off --runtime-api=on \
  python src/profile_practice.py 1 > "${COURSE_OUTPUT_DIR}/practice_step1.log" 2>&1
python src/summarize_profile.py "${COURSE_OUTPUT_DIR}/practice_step1" --blocks 36 --log "${COURSE_OUTPUT_DIR}/practice_step1.log"
```

阶段一基线耗时 **78.7150 μs**，有效带宽 **1202.24 GB/s**，实际算力 **122.77 TFLOPS**，算力利用率 **23.13%**。

## 阶段二：开启双 buffer

保持 BM=64、BN=128，将 stages 从 1 改为 2。a/b/c 各使用两份内存，尝试让不同输出块的搬运与计算并行；计算量和逻辑搬运量保持不变。

沿用 4.3 的写法，通过 `T.annotate_buffer_versions({a:s,b:s,c:s})` 显式设置输入和输出的 buffer 份数；本阶段 `s=stages=2`。

沿用阶段一的 `gemm`，只修改 `stages`：

```python
main = gemm(M=512, N=36864, K=256, BM=64, BN=128, stages=2)
kernel = tilelang.compile(main, out_idx=-1)
```

实测 kernel 中位耗时 **48.7075 μs**，相对阶段一降低 **38.1%**。

## 阶段三：调整 BM

保持 BN=128 和双 buffer，依次尝试 BM=128、256。一次计算覆盖更多 M 行，减少不同 M 分块对 B 的重复读取，同时每核循环从 64 次降为 32 次、16 次。**BM=256 时，两份结果已占用 256 KiB L0C，不能继续只增大 BM。**

继续使用阶段一的 `gemm`，保持双 buffer，分别构建两个配置：

```python
kernels = {}
for bm in (128, 256):
    main = gemm(M=512, N=36864, K=256, BM=bm, BN=128, stages=2)
    kernels[bm] = tilelang.compile(main, out_idx=-1)
```

实测 kernel 中位耗时：BM=128 为 **37.7800 μs**，BM=256 为 **31.7445 μs**；分别较前一个配置降低 **22.4%**、**16.0%**。



### 前三个阶段的完整输出校验

下面依次运行前四个配置的正确性检查。

In [2]:
import subprocess, sys
for step in range(1, 5):
    p = subprocess.run(
        [sys.executable, 'src/profile_practice.py', str(step), '--validate-only'],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print("\n".join(line for line in p.stdout.splitlines()
                       if not line.startswith("Loading tilelang libs from dev root:")))
    p.check_returncode()


practice_1 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
practice_1 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_1 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_1 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_2 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
practice_2 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_2 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_2 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_3 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
practice_3 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_3 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_3 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_4 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06


## 阶段四：A 常驻 L1 + Persistent 列优先任务分配

本阶段沿用阶段三的 BM=256、BN=128，同时调整数据复用与任务分配：将 A 的搬入移到循环外，每个核只读取一次自己负责的 256 行 A；B/C 继续采用双 buffer，并由 Persistent 按列优先分配输出任务。

M 方向有 2 个块，N 方向有 288 个块，36 个核组成 18 对。`mi=bx%2` 决定上、下半行，`bx//2` 决定起始列块。每轮列块编号增加 18，即 `ni=bx//2+it×18`；每核处理 16 个输出块，共 576 个块，覆盖完整输出且无重复。

![A 常驻 L1 与 Persistent 列优先任务分配](images/04.05_persistent_reuse.png)

### 完整实现

```python
import tilelang
import tilelang.language as T


def gemm(M=512, N=36864, K=256, BM=256, BN=128, stages=2, persistent=False):
    assert M % BM == 0 and 36 % (M // BM) == 0
    cores_n = 36 // (M // BM)
    assert N % (cores_n * BN) == 0
    assert 2 * BM * K + stages * 2 * BN * K <= 512 * 1024
    assert stages * 4 * BM * BN <= 256 * 1024

    @T.prim_func
    def main(
        A: T.Tensor((M, K), 'bfloat16'),
        B: T.Tensor((N, K), 'bfloat16'),
        C: T.Tensor((M, N), 'float32'),
    ):
        with T.Kernel(36) as bx:
            a = T.alloc_l1((BM, K), 'bfloat16')
            b = T.alloc_l1((BN, K), 'bfloat16')
            c = T.alloc_l0c((BM, BN), 'float32')
            T.annotate_buffer_versions({a: 1, b: stages, c: stages})
            mi = bx % (M // BM)
            T.copy(A[mi * BM : (mi + 1) * BM, :], a)
            for ni in T.Persistent([N // BN], cores_n, bx // (M // BM), num_stages=stages):
                T.copy(B[ni * BN : (ni + 1) * BN, :], b)
                T.gemm(a, b, c, transpose_B=True, clear_accum=True)

                T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])

    return main


main = gemm(M=512, N=36864, K=256, BM=256, BN=128, stages=2)
kernel = tilelang.compile(main, out_idx=-1)
```

与阶段三相比，BM/BN 保持不变，核心改动在两处：

1. **输入复用**：A 的 `T.copy` 移到循环外。
2. **任务分配**：循环通过 `T.Persistent([N//BN], cores_n, bx//(M//BM), num_stages=stages)` 取得列块索引，配合 `mi=bx%(M//BM)` 实现列优先任务分配。

每个核保存一份 A，B/C 各保留两份内存。通过 `T.annotate_buffer_versions({a:1,b:stages,c:stages})` 显式设置：A 常驻一份内存，B/C 各使用 `stages` 份内存；本例 `stages=2`。

保持 BM=256、BN=128 做正反序同轮对照，阶段三两次中位耗时为 **33.0380 μs、32.6525 μs**；合并优化后为 **26.0400 μs、26.8095 μs**。两次中位数的均值从 **32.8453 μs** 降至 **26.4248 μs**，降低 **19.55%**。这是 A 常驻与列优先任务分配联合调整的结果，不能只归因于其中一个因素。

### 阶段四的完整输出校验

下面的执行单元只检查正确性；性能采集使用前面的 msprof 命令。

In [1]:
import subprocess, sys
p = subprocess.run([sys.executable, 'src/profile_practice.py', '5', '--validate-only'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("\n".join(line for line in p.stdout.splitlines()
                       if not line.startswith("Loading tilelang libs from dev root:")))
p.check_returncode()

[W917 20:46:32.240594278 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-17 20:46:39  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `main` with `out_idx=[2]`
2026-09-17 20:46:51  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `main`
practice_5 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
practice_5 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_5 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
practice_5 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
Verification passed!



## 调优结果与性能对比

当前性能优化仅作为示例，有兴趣的同学可继续尝试优化，并结合有效带宽与 kernel 耗时验证效果。
下面保留课程开发时的性能对比，属于历史测量示例；当前目录不包含该轮原始采集文件，实际运行请重新采集：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">步骤</th>
<th style="text-align: left; vertical-align: top;">BM×BN</th>
<th style="text-align: left; vertical-align: top;">耗时（μs）</th>
<th style="text-align: left; vertical-align: top;">有效带宽（GB/s）</th>
<th style="text-align: left; vertical-align: top;">算力（TFLOPS）</th>
<th style="text-align: left; vertical-align: top;">利用率</th>
<th style="text-align: left; vertical-align: top;">加速比</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">单 buffer 基线</td>
<td style="text-align: left; vertical-align: top;">64×128</td>
<td style="text-align: left; vertical-align: top;">78.7150</td>
<td style="text-align: left; vertical-align: top;">1202.24</td>
<td style="text-align: left; vertical-align: top;">122.77</td>
<td style="text-align: left; vertical-align: top;">23.13%</td>
<td style="text-align: left; vertical-align: top;">1.00×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">开启双 buffer</td>
<td style="text-align: left; vertical-align: top;">64×128</td>
<td style="text-align: left; vertical-align: top;">48.7075</td>
<td style="text-align: left; vertical-align: top;">1942.90</td>
<td style="text-align: left; vertical-align: top;">198.40</td>
<td style="text-align: left; vertical-align: top;">37.38%</td>
<td style="text-align: left; vertical-align: top;">1.62×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">增大 BM</td>
<td style="text-align: left; vertical-align: top;">128×128</td>
<td style="text-align: left; vertical-align: top;">37.7800</td>
<td style="text-align: left; vertical-align: top;">2504.87</td>
<td style="text-align: left; vertical-align: top;">255.79</td>
<td style="text-align: left; vertical-align: top;">48.19%</td>
<td style="text-align: left; vertical-align: top;">2.08×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">继续增大 BM</td>
<td style="text-align: left; vertical-align: top;">256×128</td>
<td style="text-align: left; vertical-align: top;">31.7445</td>
<td style="text-align: left; vertical-align: top;">2981.11</td>
<td style="text-align: left; vertical-align: top;">304.42</td>
<td style="text-align: left; vertical-align: top;">57.35%</td>
<td style="text-align: left; vertical-align: top;">2.48×</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">A 常驻 + 列优先</td>
<td style="text-align: left; vertical-align: top;">256×128</td>
<td style="text-align: left; vertical-align: top;">26.4248</td>
<td style="text-align: left; vertical-align: top;">3581.26</td>
<td style="text-align: left; vertical-align: top;">365.71</td>
<td style="text-align: left; vertical-align: top;">68.89%</td>
<td style="text-align: left; vertical-align: top;">2.98×</td>
</tr>
</tbody>
</table>

<p><small>阶段四的19.55%降幅来自重新测量的前后对照：32.8453 μs → 26.4248 μs；表中31.7445 μs为阶段三此前测得的结果。</small></p>

计算方式沿用4.3节第3小节“性能瓶颈分析”：计算量为 9,663,676,416 FLOPs，有效数据量为 94,633,984 字节；36 核配置峰值估算为 530.8416 TFLOPS。带宽是有效带宽，利用率是按配置峰值估算的算力利用率。



## 小结

本节按四个阶段完成实践：36 核建立基线、开启双 buffer、调整 BM，最后在 BM=256、BN=128 下结合 A 常驻 L1 与 Persistent 列优先任务分配进行优化。每次修改先验证完整输出，再用相同输入比较 kernel 耗时、有效带宽与算力利用率。输入形状变化时，需要重新检查分块配置。第6章将介绍怎样定位数值错误，以及怎样用do_bench和msprof测量并读取算子耗时，为比较不同配置提供依据。

## 课后思考

1. 将 A 的搬入移到循环外，为什么仍需每个核各自保存一份 A？
2. 在 BM=256、BN=128 时，core0、core1、core2 分别负责哪些行和列块？
3. 输入 shape 改变后，需要重新检查哪些容量、任务覆盖和复用条件？

In [3]:
from pathlib import Path
print(Path("answer/04.05_answer.txt").read_text())

1. 每核有独立的 L1。A 常驻减少的是该核循环中的重复搬入，不是让所有核共享同一份 L1 数据。
2. core0 处理上半行，core1 处理下半行，两者列块为 0、18、36…270；core2 处理上半行，列块为 1、19、37…271。每核 16 个完整输出块。
3. 重新检查 L1/L0C 容量、M/N 与 tile 的整除关系、核与输出块映射，以及 A 能否在任务循环中保持复用。

